# Анализ `processors-am4.json`

Таблица: **имя**, **цена**, **ядер**, **потоков**, **частота** (ГГц), **value** = потоки × частота, **value/$** = value / цена (цена в валюте из JSON, обычно BYN).

Сортировка по убыванию **value/$**.

Зависимости: `pip install pandas` (в среде Jupyter).

Файл `processors-am4.json` ищется в текущей рабочей папке или в `onliner.by-crawler/` (удобно, если kernel стартовал из корня репозитория).

In [37]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
CANDIDATES = [
    ROOT / "processors-am4.json"
]
DATA = next((p for p in CANDIDATES if p.exists()), None)
if DATA is None:
    raise FileNotFoundError(
        "Не найден processors-am4.json. Положите его рядом с ноутбуком или запустите краулер (run-search.bat)."
    )

with open(DATA, encoding="utf-8") as f:
    rows = json.load(f)

df = pd.DataFrame(rows)
needed = ["name", "price", "coreCount", "threadCount", "maxFrequencyGHz"]
missing_cols = [c for c in needed if c not in df.columns]
if missing_cols:
    raise ValueError(f"В JSON нет колонок: {missing_cols}")

out = df[needed].copy()
out.columns = ["имя", "цена", "ядер", "потоков", "частота"]
out["value=потоков*частота"] = out["потоков"] * out["частота"]

price_num = pd.to_numeric(
    out["цена"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce",
)
out["value/руб"] = out["value=потоков*частота"].div(price_num).where(price_num > 100)

out_sorted = out.sort_values("value/руб", ascending=False, na_position="last")
out_sorted.head(12)

,имя,цена,ядер,потоков,частота,value=потоков*частота,value/руб
12,AMD Ryzen 3 3100,109.93,4,8,3.9,31.2,0.283817
32,AMD Ryzen 5 4500,245.00,6,12,4.1,49.2,0.200816
34,AMD Ryzen 5 5500,256.71,6,12,4.2,50.4,0.196330
29,AMD Ryzen 5 3600,275.05,6,12,4.2,50.4,0.183239
60,AMD Ryzen 7 5700,478.44,8,16,4.6,73.6,0.153833
67,AMD Ryzen 7 5800,494.09,8,16,4.6,73.6,0.148961
65,AMD Ryzen 7 5700X,498.15,8,16,4.6,73.6,0.147747
24,AMD Ryzen 5 3400G,231.68,4,8,4.2,33.6,0.145028
40,AMD Ryzen 5 5600,376.26,6,12,4.4,52.8,0.140328
52,AMD Ryzen 5 Pro 3600,362.73,6,12,4.2,50.4,0.138946


In [38]:
# Строки без потоков или частоты (value не посчитать)
mask = out["потоков"].isna() | out["частота"].isna()
if mask.any():
    display(out.loc[mask])
else:
    print("Все строки имеют потоки и частоту для расчёта value.")

Все строки имеют потоки и частоту для расчёта value.
